<a href="https://colab.research.google.com/github/ayeung009/APS360-Project/blob/main/Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APS360 Project: A Deep Learning Assistant for Digital Circuit Design

##Data Loading & Processing

This section involves:
1. Downloading the Roboflow Logic Gates dataset
2. Inspecting the dataset (currently it has gate bounding boxes, but **not** full Boolean expressions)
3. A manual labelling helper to attach a Boolean expression to each circuit image
4. Combining Roboflow data with self-generated (logic.ly) data
5. Tokenizing Boolean expressions into sequences
6. A PyTorch `Dataset` / `DataLoader` pipeline with train/val/test splits


### 1.1 Setup

If you're in Colab, mount your Drive so downloaded data and labels persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install roboflow torch torchvision matplotlib pandas scikit-learn --quiet


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as T

SEED = 360
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Root directory for all project data — keep this consistent across notebook sections
DATA_ROOT = Path("./data")
DATA_ROOT.mkdir(exist_ok=True)


### 1.2 Download the Roboflow dataset

Get a free API key from https://roboflow.com (Settings → API Keys) and paste it below.
This downloads the "Logic Gates Detection" dataset [2] in COCO format, which gives you
images + bounding boxes per logic gate (AND, OR, NOT, XOR, etc.) — useful context, but
you still need to derive the *overall* Boolean expression per image yourself (Section 1.4).

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="jnIBN31pWGvXkDE3MePu")
project = rf.workspace("personal-tatwd").project("logic-gates-detection")
version = project.version(12)
dataset = version.download("coco")
#this code from the website

### 1.3 Inspect the raw dataset

COCO format gives you one `_annotations.coco.json` per split (train/valid/test), each
containing `images`, `annotations` (bounding boxes + category ids), and `categories`
(the gate class names). Let's look at one example.

In [ ]:
roboflow_dir = DATA_ROOT / "/content/Logic-Gates-Detection-12" #local data root on google drive.

# COCO splits typically live in train/, valid/, test/ subfolders
split_dirs = [d for d in roboflow_dir.iterdir() if d.is_dir()]
print("Splits found:", [d.name for d in split_dirs])

with open(split_dirs[0] / "_annotations.coco.json") as f:
    coco = json.load(f)

print("Num images:", len(coco["images"]))
print("Categories:", [c["name"] for c in coco["categories"]])

In [ ]:
def show_coco_sample(coco_json_path, images_dir, image_index=0):
    with open(coco_json_path) as f:
        coco = json.load(f)

    cat_lookup = {c["id"]: c["name"] for c in coco["categories"]}
    img_info = coco["images"][image_index]
    img_path = Path(images_dir) / img_info["file_name"]
    img = Image.open(img_path).convert("RGB")

    anns = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img)
    for a in anns:
        x, y, w, h = a["bbox"]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x, y - 5, cat_lookup[a["category_id"]], color="lime", fontsize=9, weight="bold")
    ax.set_title(img_info["file_name"])
    ax.axis("off")
    plt.show()

show_coco_sample(split_dirs[0] / "_annotations.coco.json", split_dirs[0], image_index=0)


### 1.4 Manual labeling: attaching Boolean expressions

This is the step your proposal calls out explicitly: the Roboflow labels only tell you
*which gates* appear, not the resulting expression. There's no automatic way to derive
this without already solving the project — so the labels have to come from you looking
at each circuit and writing out its Boolean expression.

The helper below makes this fast: it shows one image at a time, you type the expression,
and it appends `(image_path, expression, source)` rows to a `labels.csv` so you can stop
and resume across sessions without re-labeling anything.

**Expression format convention** (keep this consistent — it will define your tokenizer's
vocabulary later):
- Variables: single uppercase letters, `A`, `B`, `C`, ...
- Operators (space-separated): `AND`, `OR`, `NOT`, `XOR`, `NAND`, `NOR`, `XNOR`
- Parentheses `(` `)` for grouping, also space-separated

Example: `( A AND B ) OR ( NOT C )`

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

LABELS_CSV = DATA_ROOT / "labels.csv"
REPO_ROOT = Path.cwd()  # assumes you're running this notebook from the repo root

def _to_relative(img_path):
    return os.path.relpath(Path(img_path).resolve(), REPO_ROOT)

def _load_labels():
    if LABELS_CSV.exists():
        return pd.read_csv(LABELS_CSV)
    return pd.DataFrame(columns=["image_path", "expression", "source"])

class InteractiveLabeler:
    def __init__(self, image_paths, source):
        self.source = source
        self.labels_df = _load_labels()
        already_done = set(self.labels_df["image_path"])
        self.queue = [p for p in image_paths if _to_relative(p) not in already_done]
        self.index = 0

        self.text = widgets.Text(placeholder="e.g. ( A AND B ) OR C", description="Expr:")
        self.save_btn = widgets.Button(description="Save & Next", button_style="success")
        self.skip_btn = widgets.Button(description="Skip", button_style="warning")
        self.stop_btn = widgets.Button(description="Stop", button_style="danger")
        self.output = widgets.Output()

        self.text.on_submit(lambda w: self._save())       # Enter key = save
        self.save_btn.on_click(lambda b: self._save())
        self.skip_btn.on_click(lambda b: self._advance())
        self.stop_btn.on_click(lambda b: self._stop())

        display(widgets.VBox([
            self.output,
            self.text,
            widgets.HBox([self.save_btn, self.skip_btn, self.stop_btn]),
        ]))
        self._render()

    def _render(self):
        with self.output:
            clear_output(wait=True)
            if self.index >= len(self.queue):
                print(f"All done! Total labeled so far: {len(self.labels_df)}")
                return
            img_path = self.queue[self.index]
            img = Image.open(img_path).convert("RGB")
            plt.figure(figsize=(5, 5))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"{Path(img_path).name}  ({self.index + 1}/{len(self.queue)})")
            plt.show()
        self.text.value = ""

    def _save(self):
        if self.index >= len(self.queue):
            return
        expr = self.text.value.strip()
        if expr:
            rel_path = _to_relative(self.queue[self.index])
            self.labels_df = pd.concat([
                self.labels_df,
                pd.DataFrame([{"image_path": rel_path, "expression": expr, "source": self.source}])
            ], ignore_index=True)
            self.labels_df.to_csv(LABELS_CSV, index=False)  # save after every label
        self._advance()

    def _advance(self):
        self.index += 1
        self._render()

    def _stop(self):
        with self.output:
            clear_output(wait=True)
            print(f"Stopped. Total labeled so far: {len(self.labels_df)}")
        self.index = len(self.queue)

In [ ]:
import os

roboflow_dir = DATA_ROOT / "/content/Logic-Gates-Detection-12"
train_dir = roboflow_dir / "train"

print(os.listdir(train_dir)[:10])  # confirm actual filenames/extension (.jpg vs .png)

In [ ]:
all_roboflow_images = list(train_dir.glob("*.jpg"))  # match the extension you confirmed earlier
print(f"Found {len(all_roboflow_images)} images")

labeler = InteractiveLabeler(all_roboflow_images, source="roboflow")

In [ ]:
print(LABELS_CSV.resolve())        # full absolute path on disk
print(LABELS_CSV.exists())         # should print True once you've saved at least one label

In [ ]:
!pip install schemdraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.5/152.5 kB 6.3 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade schemdraw

In [ ]:
import os
import random
import pandas as pd
from tqdm import tqdm
from schemdraw.parsing import logicparse

# =====================================================
# Configuration
# =====================================================

NUM_SAMPLES = 1000
OUTPUT_DIR = "logic_dataset"

IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

random.seed(42)

VARIABLES = ["A", "B", "C", "D", "E"]

BINARY_OPS = [
    "and",
    "or",
    "xor",
    "nand",
    "nor",
    "xnor"
]

# =====================================================
# Boolean Expression Generator
# =====================================================

def generate_expression(variables):
    """
    Generate a random Boolean expression where every
    variable is used exactly once.
    """

    # Start with each input as its own expression
    expressions = variables.copy()

    random.shuffle(expressions)

    # Single input case
    if len(expressions) == 1:

        if random.random() < 0.5:
            return f"(not {expressions[0]})"

        return expressions[0]


    # Randomly combine expressions until one remains
    while len(expressions) > 1:

        # Pick two sub-expressions
        index1 = random.randrange(len(expressions))
        a = expressions.pop(index1)

        index2 = random.randrange(len(expressions))
        b = expressions.pop(index2)


        # Random NOT gates
        if random.random() < 0.25:
            a = f"(not {a})"

        if random.random() < 0.25:
            b = f"(not {b})"


        # Combine with random gate
        op = random.choice(BINARY_OPS)

        combined = f"({a} {op} {b})"


        # Occasionally invert gate output
        if random.random() < 0.15:
            combined = f"(not {combined})"


        expressions.append(combined)


    return expressions[0]


# =====================================================
# Dataset Generation
# =====================================================

records = []
seen = set()

counter = 0

print("Generating logic circuit dataset...")

with tqdm(total=NUM_SAMPLES) as pbar:

    while counter < NUM_SAMPLES:

        # Pick number of inputs (1-5)
        n_inputs = random.randint(1, 5)

        # Select inputs
        vars_used = VARIABLES[:n_inputs]


        # Generate expression
        expression = generate_expression(vars_used)


        # Avoid duplicate labels
        if expression in seen:
            continue

        seen.add(expression)


        # Save image
        filename = f"{counter:05d}.png"
        filepath = os.path.join(IMAGE_DIR, filename)


        drawing = logicparse(expression)

        drawing.save(
            filepath,
            dpi=300
        )


        records.append({
            "filename": filename,
            "expression": expression,
            "n_inputs": n_inputs
        })


        counter += 1
        pbar.update(1)


# =====================================================
# Save Labels
# =====================================================

labels_path = os.path.join(
    OUTPUT_DIR,
    "labels.csv"
)

df = pd.DataFrame(records)

df.to_csv(
    labels_path,
    index=False
)


print("\nDataset complete!")
print(f"Generated {NUM_SAMPLES} unique circuits")
print(f"Images saved to: {IMAGE_DIR}")
print(f"Labels saved to: {labels_path}")

Generating logic circuit dataset...


100%|██████████| 1000/1000 [02:06<00:00,  7.89it/s]


Dataset complete!
Generated 1000 unique circuits
Images saved to: logic_dataset/images
Labels saved to: logic_dataset/labels.csv


training part

In [ ]:
!pip install transformers torch torchvision pandas pillow tqdm scikit-learn

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from sklearn.model_selection import train_test_split

from transformers import ViTModel, ViTConfig

from tqdm import tqdm

In [ ]:
DATA_DIR = "logic_dataset"

IMAGE_DIR = os.path.join(DATA_DIR,"images")

labels = pd.read_csv(
    os.path.join(DATA_DIR,"labels.csv")
)

labels.head()

In [ ]:
train_df, temp_df = train_test_split(
    labels,
    test_size=0.3,
    random_state=42
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)


print(len(train_df))
print(len(val_df))
print(len(test_df))

expression tokenizer!

In [ ]:
SPECIAL = [
    "<pad>",
    "<bos>",
    "<eos>"
]


TOKENS = [
    "A","B","C","D","E",
    "and",
    "or",
    "xor",
    "nand",
    "nor",
    "xnor",
    "not",
    "(",
    ")"
]


vocab = SPECIAL + TOKENS


stoi = {
    token:i
    for i,token in enumerate(vocab)
}


itos = {
    i:t
    for t,i in stoi.items()
}


VOCAB_SIZE = len(vocab)

PAD_IDX = stoi["<pad>"]
BOS_IDX = stoi["<bos>"]
EOS_IDX = stoi["<eos>"]

In [ ]:
def tokenize(expr):

    expr = (
        expr.replace("("," ( ")
            .replace(")"," ) ")
    )

    tokens = expr.split()

    ids = [
        BOS_IDX
    ]

    for t in tokens:
        ids.append(stoi[t])

    ids.append(EOS_IDX)

    return ids

dataset class

In [ ]:
class LogicDataset(Dataset):

    def __init__(self, dataframe):

        self.df=dataframe

        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
        ])


    def __len__(self):
        return len(self.df)


    def __getitem__(self,idx):

        row=self.df.iloc[idx]


        img=Image.open(
            os.path.join(
                IMAGE_DIR,
                row.filename
            )
        )


        img=self.transform(img)


        label=torch.tensor(
            tokenize(row.expression),
            dtype=torch.long
        )


        return img,label

collate function

In [ ]:
from torch.nn.utils.rnn import pad_sequence


def collate(batch):

    images,labels=zip(*batch)


    images=torch.stack(images)


    labels=pad_sequence(
        labels,
        batch_first=True,
        padding_value=PAD_IDX
    )


    return images,labels

dataloaders


In [ ]:
train_loader=DataLoader(
    LogicDataset(train_df),
    batch_size=16,
    shuffle=True,
    collate_fn=collate
)


val_loader=DataLoader(
    LogicDataset(val_df),
    batch_size=16,
    collate_fn=collate
)


test_loader=DataLoader(
    LogicDataset(test_df),
    batch_size=16,
    collate_fn=collate
)

VIT encoder

In [ ]:
encoder = ViTModel.from_pretrained(
    "google/vit-base-patch16-224"
)

decoder

In [ ]:
class ExpressionDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding=nn.Embedding(
            VOCAB_SIZE,
            768
        )


        layer=nn.TransformerDecoderLayer(
            d_model=768,
            nhead=8,
            batch_first=True
        )


        self.decoder=nn.TransformerDecoder(
            layer,
            num_layers=4
        )


        self.fc=nn.Linear(
            768,
            VOCAB_SIZE
        )


    def forward(
        self,
        memory,
        tokens
    ):


        x=self.embedding(tokens)


        out=self.decoder(
            x,
            memory
        )


        return self.fc(out)

full model

In [ ]:
class ViTLogicModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder=encoder

        self.decoder=ExpressionDecoder()


    def forward(self,img,tokens):

        features=self.encoder(
            img
        ).last_hidden_state


        return self.decoder(
            features,
            tokens
        )

Training

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"


model=ViTLogicModel().to(device)


optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)


criterion=nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

In [ ]:
EPOCHS=20


for epoch in range(EPOCHS):

    model.train()

    total_loss=0


    for imgs,labels in tqdm(train_loader):

        imgs=imgs.to(device)
        labels=labels.to(device)


        inp=labels[:,:-1]
        target=labels[:,1:]


        pred=model(
            imgs,
            inp
        )


        loss=criterion(
            pred.reshape(-1,VOCAB_SIZE),
            target.reshape(-1)
        )


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        total_loss+=loss.item()


    print(
        "Epoch",
        epoch,
        "loss",
        total_loss/len(train_loader)
    )